# ComplianceGPT — Single Run

In [4]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!pip install bitsandbytes accelerate sentence-transformers faiss-cpu pyyaml

In [ ]:
# Run one query end-to-end (pipeline import, retrieval, contract, verification)

from pipeline import ComplianceGPTPipeline

# --- Paths (edit for your repo) ---
REV5_CCS_PATH = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
REV4_CCS_PATH = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"

# If you're running in this chat sandbox, you can use:
# REV5_CCS_PATH = "/mnt/data/NIST_SP-800-53_rev5_catalog.jsonl"
# REV4_CCS_PATH = "/mnt/data/NIST_SP-800-53_rev4_catalog.jsonl"

DOC_FILTER_MODE = "all"  # "all" (recommended) or "prefer_statement_only"
STRICT_CCS_ASSERT = True

query = "How must a system be configured to handle consecutive invalid logon attempts?"
version = "rev5"  # "rev4" or "rev5"
ccs_path = REV5_CCS_PATH if version == "rev5" else REV4_CCS_PATH

pipe = ComplianceGPTPipeline(
    framework_version=version,
    doc_filter_mode=DOC_FILTER_MODE,
    ccs_path=ccs_path,
    strict_ccs_assert=STRICT_CCS_ASSERT,
)

out = pipe.answer(query, odp_policy="ASK")  # ASK keeps placeholders verbatim
contract = out.get("contract", {}) or {}

print("STATUS:", contract.get("status"))
print("PRIMARY CITATION:", contract.get("primary_citation"))
print("\nANSWER (with citation):\n", contract.get("answer_text_with_citation", contract.get("answer_text","")))
print("\nODP/PRM required list:", contract.get("odp_required_list", []))
print("\nVerifier errors:", contract.get("verifier_errors", []))
